In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt

import seaborn as sns

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')

In [ ]:
print('First 5 Rows Train Data Set')
train_df.head()

In [ ]:
print('Train Data Set Information')
train_df.info()
print('\nTest Data Set Information')
test_df.info()

In [ ]:
print('Train Data Set Numerical Summary')
train_df.describe()

- PassengerId: It is just an ID number, statistically meaningless. We probably won't use it in modeling.
- Survived: Our target variable is (0 or 1). Its mean of 0.38 indicates that approximately 38% of passengers survived. (We will examine this in more detail shortly).
- Pclass: It takes values such as 1, 2, 3. Although it appears to be numerical, it is actually a categorical variable (1st class, 2nd class, 3rd class). Its average (2.3) is not very meaningful, but it indicates that passengers are mostly concentrated in the lower classes (close to 3rd class).
- Age: We already knew there were significant missing data (count 714). We see that the distribution ranges from 0.42 (infant) to 80 (elderly).
- SibSp & Parch: Most passengers (median = 0, 75% = 1 or 0) travel alone or with at most one family member.
- Fare: There is significant skewness. There is a large difference between the mean (32) and the median (14.4), and the maximum value (512) is very high. This indicates that a small group purchased very expensive tickets, while the majority purchased inexpensive tickets.

In [ ]:
print('Train Data Set Categoric Summary')
train_df.describe(include=['object'])

- Name: All 891 passengers have different names (unique=891). This column is meaningless for modeling in its current form. However, during the “Feature Engineering” phase, we can transform it into useful information by removing titles such as “Mr.”, “Mrs.”, and “Miss” from the names.
- Sex: The majority of passengers are male (577 / 891).
- Ticket: Ticket numbers. There are too many unique values (unique=681). Like Name, this is meaningless as it stands, but perhaps a pattern can be found in the future.
- Cabin: There is too much missing data (count=204) and too many unique values (unique=147). We will need to decide how to handle this column (perhaps delete it).
- Embarked: Port of embarkation. There are 3 different ports (unique=3). Most embarkations were from the ‘S’ (Southampton) port (top=‘S’). We knew from .info() that there were 2 missing data points.

In [ ]:
print('Survived Distribution (Numeric)')
print(train_df['Survived'].value_counts())

print('Survived Distribution (Percentage)')
print(train_df['Survived'].value_counts(normalize=True))

- This is close to a situation called an “Imbalanced Dataset” but not at a critical level. If the survivors were 10% or less, we might have needed to use special techniques (SMOTE, oversampling, undersampling, etc.) during the modeling phase. For now, this distribution is acceptable for working with standard algorithms. 

In [ ]:
train_ids = train_df['PassengerId']
test_ids = test_df['PassengerId']
y_train = train_df['Survived']

train_df = train_df.drop(columns=['PassengerId', 'Survived'])
test_df = test_df.drop(columns=['PassengerId'])

df = pd.concat([train_df, test_df], keys=['train', 'test'])

print('Concat Data Set First 5 Rows')
df.head()

In [ ]:
df = df.drop(columns=['Ticket', 'Cabin'])
df.info()

In [ ]:
embarked_mode = df.loc['train', 'Embarked'].mode()[0]
print(f'The most frequent port of embarkation in the training data (Mode): {embarked_mode}')

df['Embarked'].fillna(embarked_mode, inplace=True)

df.info()

In [ ]:
fare_median = df.loc['train', 'Fare'].median()

In [ ]:
df['Fare'].fillna(fare_median, inplace=True)

In [ ]:
age_median = df.loc['train', 'Age'].median()

df['Age'].fillna(age_median, inplace=True)

df.info()

### Data Preparation

In [ ]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

print(df['Title'].value_counts())

In [ ]:
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 
                                     'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

print(df['Title'].value_counts())

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

print(f"\nFamily Size Distribution:\n{df[['FamilySize', 'SibSp', 'Parch']].head()}")

In [ ]:
df = df.drop(columns=['Name', 'SibSp', 'Parch'])

df.head()

#### Convertion of Categorical Variables

In [ ]:
df['Sex'] = df['Sex'].map({'male':0, 'female':1})

df['Pclass'] = df['Pclass'].astype(str)

ohe_cols = ['Pclass', 'Embarked', 'Title']
df = pd.get_dummies(df, columns = ohe_cols)

df.head()

In [ ]:
X_train = df.loc['train']
X_test = df.loc['test']

print(f'X_train Size: {X_train.shape}')
print(f'X_test Size: {X_test.shape}')
print(f'y_train Size: {y_train.shape}')

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
print(f'Split Data Size: {X_train_split.shape}')
print(f'Validation (X_val) Size: {X_val.shape}')

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model_lr = LogisticRegression(max_iter=1000)

model_lr.fit(X_train_split, y_train_split)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
y_train_preds = model_lr.predict(X_train_split)

y_val_preds = model_lr.predict(X_val)

train_accuracy = accuracy_score(y_train_split, y_train_preds)

val_accuracy = accuracy_score(y_val, y_val_preds)

print(f'Train Accuracy: {train_accuracy:.4f}')

print(f'Validation Accuracy: {val_accuracy:.4f}')

if train_accuracy > val_accuracy + 0.05:
    print("\Result: The model shows a overfitting.")
elif train_accuracy < 0.7 or val_accuracy < 0.7:
    print("\Result: The model shows a underfitting.")
else:
    print("\Result: The model appears robust.")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
cm_lr = confusion_matrix(y_val, y_val_preds)

print(f'Logistic Regression Confusion Matrix')
print(cm_lr)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Non_Survived (0)', 'Survived (1)'])

disp.plot(cmap=plt.cm.Blues)
plt.show()

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
report = classification_report(y_val, y_val_preds, target_names=['Non_Survived (0)', 'Survived (1)'])
print(report)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
model_dt_overfit = DecisionTreeClassifier(random_state=42)

model_dt_overfit.fit(X_train_split, y_train_split)

In [ ]:
y_train_preds_dt = model_dt_overfit.predict(X_train_split)

y_val_preds_dt = model_dt_overfit.predict(X_val)

train_accuracy_dt = accuracy_score(y_train_split, y_train_preds_dt)
val_accuracy_dt = accuracy_score(y_val, y_val_preds_dt)


print('Descision Tree')
print(f' Train Accuracy {train_accuracy_dt:.4f}')
print(f' Validation Accuracy {val_accuracy_dt:.4f}')

print('Logistic Regression')
print(f'Previous Train Accuracy {train_accuracy:.4f}')
print(f'Previous Validation Accuracy {val_accuracy:.4f}')

In [ ]:
model_dt_pruned = DecisionTreeClassifier(max_depth=5, random_state=42)

model_dt_pruned.fit(X_train_split, y_train_split)

y_train_preds_pruned = model_dt_pruned.predict(X_train_split)
y_val_preds_pruned = model_dt_pruned.predict(X_val)

train_accuracy_pruned = accuracy_score(y_train_split, y_train_preds_pruned)
val_accuracy_pruned = accuracy_score(y_val, y_val_preds_pruned)

print('Decision Tree Result (Pruned')
print(f'Train Accuracy {train_accuracy_pruned:.4f}')
print(f'Validation Accuracy {val_accuracy_pruned:.4f}')

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

model_rf.fit(X_train_split, y_train_split)

In [ ]:
y_train_preds_rf = model_rf.predict(X_train_split)

y_val_preds_rf = model_rf.predict(X_val)

train_accuracy_rf = accuracy_score(y_train_split, y_train_preds_rf)
val_accuracy_rf = accuracy_score(y_val, y_val_preds_rf)

print('Random Forest')
print(f'Train Accuracy {train_accuracy_rf:.4f}')
print(f'Validation Accuracy {val_accuracy_rf:.4f}')

print('Comparison with Previous Models ')
print(f'Logistic Regression: {val_accuracy:.4f}')
print(f'Pruned Decision Tree: {val_accuracy_pruned:.4f}')

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
cv_scores_rf = cross_val_score(
    model_rf,
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

print('Cross Validation Complated')
print(f"5 Different Accuracy Score: {cv_scores_rf}")
print(f"Mean of Scores: {cv_scores_rf.mean():.4f}")
print(f"Standard Deviation of Scores: {cv_scores_rf.std():.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [100,150,200],
    'max_depth': [5,10,None],
    'min_samples_leaf': [1,2,4]
}

rf_base = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid = param_grid,
    cv = 5, 
    scoring = 'accuracy',
    n_jobs = 1
)

grid_search.fit(X_train, y_train)

In [ ]:
print(f"Best Average Cross-Validation Score: {grid_search.best_score_:.4f}")

print(f"Best Hyperparameters: {grid_search.best_params_}")

final_model = grid_search.best_estimator_

print(f"\nDefault Model Score: {cv_scores_rf.mean():.4f}") 
print(f"Optimized Model Score : {grid_search.best_score_:.4f}")

In [ ]:
test_predictions = final_model.predict(X_test)

submission_df = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": test_predictions
})

submission_df.to_csv('submission.csv', index=False)

print(submission_df.head())